In [1]:
# ════════════════════════════════════════════
# CELL 1 — Setup
# ════════════════════════════════════════════
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import cv2
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from src.phase2_sfm.incremental_sfm  import IncrementalSfM
from src.phase2_sfm.visualize        import visualize_reconstruction
from src.evaluation.compare          import compare_camera_trajectories
from src.utils.io_utils              import export_ply
from src.utils.camera                import estimate_intrinsics_from_fov

print("Imports OK")
os.makedirs("../outputs/sparse", exist_ok=True)

Imports OK


In [2]:
# ════════════════════════════════════════════
# CELL 2 — Camera Intrinsics Setup
# ════════════════════════════════════════════
# Option A: Known intrinsics (best)
K = np.array([
    [800.0,   0.0, 320.0],
    [  0.0, 800.0, 240.0],
    [  0.0,   0.0,   1.0]
], dtype=np.float64)

# Option B: Estimate from FOV
# K = estimate_intrinsics_from_fov(640, 480, fov_horizontal_deg=60)

# Option C: Auto-detect from images
IMAGE_DIR = "../data/raw"
image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
]) if os.path.isdir(IMAGE_DIR) else []

if image_files:
    sample = cv2.imread(os.path.join(IMAGE_DIR, image_files[0]))
    h, w = sample.shape[:2]
    focal = max(h, w) * 1.2
    K = np.array([[focal, 0, w/2], [0, focal, h/2], [0, 0, 1]], dtype=np.float64)
    print(f"Auto-detected intrinsics for {w}x{h} images, focal={focal:.1f}px")
else:
    print(f"Using default intrinsics:\n{K}")

Auto-detected intrinsics for 1600x1066 images, focal=1920.0px


In [3]:


# ════════════════════════════════════════════
# CELL 3 — Run Incremental SfM
# ════════════════════════════════════════════
sfm = IncrementalSfM(K, config={
    "detector":         "sift",
    "max_keypoints":    6000,
    "matcher":          "flann",
    "ratio_threshold":  0.75,
    "ransac_threshold": 1.0,
    "min_matches":      25,
})

if image_files:
    sfm.load_images(IMAGE_DIR, max_dim=1024)
    sfm.reconstruct(bundle_adjust_interval=3)
else:
    # ── Synthetic demo (no real images needed) ──────────────
    print("No images found. Running synthetic SfM demo...")

    def _make_synthetic_sfm():
        """Build a tiny synthetic SfM reconstruction for demo."""
        from src.phase2_sfm.triangulate import triangulate_points
        from src.utils.transforms       import rotation_matrix_y

        n_pts = 150
        np.random.seed(42)
        pts3d = np.random.randn(n_pts, 3)
        pts3d[:, 2] += 5.0

        sfm_demo = IncrementalSfM(K)
        sfm_demo.images      = [np.zeros((480, 640, 3), np.uint8)] * 6
        sfm_demo.image_names = [f"frame_{i:03d}.jpg" for i in range(6)]

        R_list = [rotation_matrix_y(i * 15.0) for i in range(6)]
        t_list = [np.array([[np.sin(np.radians(i * 15)) * 0.3], [0], [0]]) for i in range(6)]

        for i, (R, t) in enumerate(zip(R_list, t_list)):
            sfm_demo.registered_cameras[i] = (R, t)

        sfm_demo.points_3d    = list(pts3d)
        sfm_demo.point_colors = list(
            (np.random.rand(n_pts, 3) * 255).astype(np.uint8)
        )
        return sfm_demo

    sfm = _make_synthetic_sfm()

Loading 40 images from ../data/raw


100%|█████████████████████████████████████████████████████████| 40/40 [00:01<00:00, 32.24it/s]


Loaded 40 images
Detecting features...


  2%|█▍                                                        | 1/40 [00:00<00:05,  6.52it/s]

  image_0000.jpg: 6000 features


...

 98%|███████████████████████████████████████████████████████▌ | 39/40 [00:06<00:00,  6.97it/s]

  image_0037.jpg: 6000 features
  image_0038.jpg: 2347 features


100%|█████████████████████████████████████████████████████████| 40/40 [00:06<00:00,  6.18it/s]

  image_0039.jpg: 6000 features
Matching features...


  image_0000.jpg <-> image_0001.jpg: 356/422 inliers (84.4%)
  image_0000.jpg <-> image_0004.jpg: 853/937 inliers (91.0%)
  image_0000.jpg <-> image_0005.jpg: 127/211 inliers (60.2%)
  image_0000.jpg <-> image_0009.jpg: 63/114 inliers (55.3%)
  image_0000.jpg <-> image_0013.jpg: 87/151 inliers (57.6%)
  image_0001.jpg <-> image_0002.jpg: 261/296 inliers (88.2%)
  image_0001.jpg <-> image_0004.jpg: 105/175 inliers (60.0%)
  image_0001.jpg <-> image_0005.jpg: 316/409 inliers (77.3%)
  image_0001.jpg <-> image_0006.jpg: 130/216 inliers (60.2%)
  image_0001.jpg <-> image_0009.jpg: 94/134 inliers (70.1%)
  image_0001.jpg <-> image_0010.jpg: 35/71 inliers (49.3%)
  image_0001.jpg <-> image_0013.jpg: 30/64 inliers (46.9%)
  image_0001.jpg <-> image_0014.jpg: 29/55 inliers (52.7%)
  image_0002.jpg <-> image_0003.jpg: 1048/1132 inliers (92.6%)
  image_0002.jpg <-> image_0005.jpg: 70/101 inliers (69.3%)
  image_0002.jpg <-> image_0006.jpg: 270/378 inliers (71.4%)
  image_0002.jpg <-> image_0007.

In [4]:
# ════════════════════════════════════════════
# CELL 4 — Results Summary
# ════════════════════════════════════════════
points, colors = sfm.get_point_cloud()
poses          = sfm.get_camera_poses()

print(f"\n{'='*50}")
print(f"SfM Reconstruction Results")
print(f"{'='*50}")
print(f"  Registered cameras : {len(poses)}")
print(f"  3D points          : {len(points):,}")
if image_files:
    mean_err = sfm.compute_mean_reprojection_error()
    print(f"  Mean reprojection  : {mean_err:.3f} px")
print(f"{'='*50}")

# Bounding box
if len(points) > 0:
    print(f"\n  Point cloud bounds:")
    print(f"    X: [{points[:,0].min():.2f}, {points[:,0].max():.2f}]")
    print(f"    Y: [{points[:,1].min():.2f}, {points[:,1].max():.2f}]")
    print(f"    Z: [{points[:,2].min():.2f}, {points[:,2].max():.2f}]")


SfM Reconstruction Results
  Registered cameras : 40
  3D points          : 21,491
  Mean reprojection  : 0.684 px

  Point cloud bounds:
    X: [-135.44, 87.31]
    Y: [-282.40, 3.17]
    Z: [0.20, 1014.12]


In [5]:
# ════════════════════════════════════════════
# CELL 5 — Visualize Point Cloud (2D Projections)
# ════════════════════════════════════════════
if len(points) > 0:
    # Get camera centers
    centers = np.array([-R.T @ t.ravel() for R, t in poses.values()])

    fig = plt.figure(figsize=(18, 6))

    view_specs = [
        ("Top View (X-Z)",  0, 2, "X", "Z"),
        ("Side View (Y-Z)", 1, 2, "Y", "Z"),
        ("Front View (X-Y)",0, 1, "X", "Y"),
    ]

    for plot_i, (title, ax_x, ax_y, xlabel, ylabel) in enumerate(view_specs):
        ax = fig.add_subplot(1, 3, plot_i + 1)

        # Subsample for display
        n_disp = min(10_000, len(points))
        idx    = np.random.choice(len(points), n_disp, replace=False)
        c_disp = colors[idx] if len(colors) == len(points) else np.ones((n_disp, 3)) * 0.5

        ax.scatter(
            points[idx, ax_x], points[idx, ax_y],
            c=c_disp, s=0.5, alpha=0.6
        )
        if len(centers) > 0:
            ax.scatter(
                centers[:, ax_x], centers[:, ax_y],
                c="red", s=80, marker="^", zorder=5, label="Cameras"
            )
            ax.plot(
                centers[:, ax_x], centers[:, ax_y],
                "r-", linewidth=1, alpha=0.5
            )
        ax.set_title(title, fontsize=12)
        ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
        ax.set_aspect("equal", "box")
        ax.legend(markerscale=2, fontsize=9)
        ax.grid(True, alpha=0.2)

    plt.suptitle(
        f"SfM Sparse Reconstruction — {len(points):,} points, {len(poses)} cameras",
        fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("../outputs/sparse/sfm_2d_views.png", dpi=150)
    plt.show()
    print("Saved: outputs/sparse/sfm_2d_views.png")

Saved: outputs/sparse/sfm_2d_views.png


C:\Users\Harsh\AppData\Local\Temp\ipykernel_9632\1699265135.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:


# ════════════════════════════════════════════
# CELL 6 — 3D Visualization (matplotlib)
# ════════════════════════════════════════════
if len(points) > 0:
    fig = plt.figure(figsize=(12, 9))
    ax  = fig.add_subplot(111, projection="3d")

    n_disp = min(8_000, len(points))
    idx    = np.random.choice(len(points), n_disp, replace=False)
    c_disp = colors[idx] if len(colors) == len(points) else np.ones((n_disp, 3)) * 0.5

    ax.scatter(
        points[idx, 0], points[idx, 1], points[idx, 2],
        c=c_disp, s=1, alpha=0.6
    )

    if len(centers) > 0:
        ax.scatter(
            centers[:, 0], centers[:, 1], centers[:, 2],
            c="red", s=100, marker="^", zorder=10, label="Cameras"
        )
        ax.plot(
            centers[:, 0], centers[:, 1], centers[:, 2],
            "r-", linewidth=2, alpha=0.7
        )

    ax.set_title(
        f"SfM 3D Point Cloud\n{len(points):,} points | {len(poses)} cameras",
        fontsize=13
    )
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("../outputs/sparse/sfm_3d.png", dpi=150)
    plt.show()
    print("Saved: outputs/sparse/sfm_3d.png")


Saved: outputs/sparse/sfm_3d.png


C:\Users\Harsh\AppData\Local\Temp\ipykernel_9632\1663796556.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:

# ════════════════════════════════════════════
# CELL 7 — Export Point Cloud
# ════════════════════════════════════════════
ply_path = "../outputs/sparse/point_cloud.ply"
export_ply(ply_path, points, colors if len(colors) == len(points) else None)

print(f"\nExported {len(points):,} points to {ply_path}")
print("Open in MeshLab, CloudCompare, or Blender for 3D inspection.")

Saved PLY: ../outputs/sparse/point_cloud.ply (21491 points, 0 faces)

Exported 21,491 points to ../outputs/sparse/point_cloud.ply
Open in MeshLab, CloudCompare, or Blender for 3D inspection.


In [8]:


# ════════════════════════════════════════════
# CELL 8 — Reprojection Error Analysis
# ════════════════════════════════════════════
if image_files and len(sfm.observations) > 0:
    from src.phase2_sfm.triangulate import compute_reprojection_error

    pts_3d = np.array(sfm.points_3d)
    errors_per_camera = {}

    for cam_idx, pt_idx, pt_2d in sfm.observations:
        if cam_idx not in sfm.registered_cameras:
            continue
        if pt_idx >= len(pts_3d):
            continue
        R, t = sfm.registered_cameras[cam_idx]
        errs = compute_reprojection_error(
            pts_3d[pt_idx:pt_idx+1], pt_2d.reshape(1, 2), K, R, t
        )
        errors_per_camera.setdefault(cam_idx, []).extend(errs.tolist())

    all_errors = [e for errs in errors_per_camera.values() for e in errs]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(all_errors, bins=50, color="steelblue", edgecolor="none")
    axes[0].axvline(np.mean(all_errors),   color="red",   linestyle="--", label=f"Mean={np.mean(all_errors):.2f}px")
    axes[0].axvline(np.median(all_errors), color="orange",linestyle="--", label=f"Median={np.median(all_errors):.2f}px")
    axes[0].set_xlabel("Reprojection Error (pixels)"); axes[0].set_ylabel("Count")
    axes[0].set_title("Reprojection Error Distribution"); axes[0].legend()

    cam_means = {k: np.mean(v) for k, v in errors_per_camera.items()}
    axes[1].bar(list(cam_means.keys()), list(cam_means.values()), color="steelblue")
    axes[1].axhline(np.mean(all_errors), color="red", linestyle="--", label=f"Global mean={np.mean(all_errors):.2f}px")
    axes[1].set_xlabel("Camera Index"); axes[1].set_ylabel("Mean Error (px)")
    axes[1].set_title("Per-Camera Reprojection Error"); axes[1].legend()

    plt.suptitle("SfM Quality Analysis", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/sparse/reprojection_error.png", dpi=150)
    plt.show()
    print(f"\nMean reprojection error: {np.mean(all_errors):.3f} px")
    print(f"Max  reprojection error: {np.max(all_errors):.3f} px")
    print("Good reconstruction: < 2.0 px. Excellent: < 1.0 px.")


Mean reprojection error: 0.684 px
Max  reprojection error: 6.612 px
Good reconstruction: < 2.0 px. Excellent: < 1.0 px.


C:\Users\Harsh\AppData\Local\Temp\ipykernel_9632\2792188402.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
